In [1]:
import torch
import gc

# Очищаем кэш GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


# Принудительный сборщик мусора
gc.collect()

# Проверяем результат
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 0.00 GB
GPU memory cached: 0.00 GB


In [2]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch.nn as nn

import torch_pruning as tp


RESULTS_DIR = "LORA_gemma/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE = 2
SEQ_LENGTH = 64
# MODEL_NAME = "google/gemma-2-2b"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    attn_implementation="sdpa"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded")

Model loaded


In [ ]:
# # === LoRA CONFIG ===
# lora_config = LoraConfig(
#     r=8,  # rank
#     lora_alpha=16,
#     target_modules=[
#         "q_proj",
#         "k_proj",
#         "v_proj",
#         "o_proj",
#         "up_proj",
#         "down_proj"
#     ],
#     lora_dropout=0.0,
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# # === APPLY LoRA ===
# model = get_peft_model(model, lora_config)
# for name, param in model.named_parameters():
#     if "lora_" in name:
#         param.requires_grad = True
#     else:
#         param.requires_grad = False

# print("\nLoRA applied")
# model.print_trainable_parameters()


LoRA applied
trainable params: 3,293,184 || all params: 497,325,952 || trainable%: 0.6622


In [4]:
example_inputs = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device),
    "attention_mask": torch.ones(BATCH_SIZE, SEQ_LENGTH).to(device),
    "labels": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device)
}

# SDPA PRUNER

In [8]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch.nn as nn
import torch.nn.functional as F
from typing import Sequence, List

import torch_pruning as tp
from torch_pruning.pruner.function import BasePruningFunc
from torch_pruning import ops


In [9]:
class SDPAPruner(BasePruningFunc):
    """
    Pruner for MultiheadAttention with SDPA for Qwen2 models.
    Handles the specific structure of Qwen2Attention.
    """
    
    TARGET_MODULES = (nn.Module,)
    
    def __init__(self, pruning_dim: int = 1):
        super().__init__(pruning_dim=pruning_dim)
    
    def check(self, layer: nn.Module, idxs: Sequence[int], to_output: bool = True) -> None:
        """Validate pruning indices."""
        super().check(layer, idxs, to_output)
        
        # Check if this is Qwen2Attention
        if hasattr(layer, 'num_heads') and hasattr(layer, 'head_dim'):
            embed_dim = layer.num_heads * layer.head_dim
            new_embed_dim = embed_dim - len(idxs)
            assert new_embed_dim % layer.num_heads == 0, \
                f"embed_dim ({new_embed_dim}) must be divisible by num_heads ({layer.num_heads})"
    
    def prune_out_channels(self, layer: nn.Module, idxs: Sequence[int]) -> nn.Module:
        """Prune output channels (embedding dimension)."""
        # Get current dimensions
        num_heads = layer.num_heads
        head_dim = layer.head_dim
        embed_dim = num_heads * head_dim
        
        keep_idxs = list(set(range(embed_dim)) - set(idxs))
        keep_idxs.sort()
        new_embed_dim = len(keep_idxs)
        new_num_heads = new_embed_dim // head_dim
        
        # Prune q_proj (query projection)
        if hasattr(layer, 'q_proj') and layer.q_proj is not None:
            layer.q_proj = self._prune_linear(layer.q_proj, keep_idxs, is_output=True)
        
        # Prune k_proj (key projection)
        if hasattr(layer, 'k_proj') and layer.k_proj is not None:
            layer.k_proj = self._prune_linear(layer.k_proj, keep_idxs, is_output=True)
        
        # Prune v_proj (value projection)
        if hasattr(layer, 'v_proj') and layer.v_proj is not None:
            layer.v_proj = self._prune_linear(layer.v_proj, keep_idxs, is_output=True)
        
        # Prune o_proj (output projection) - prune both dimensions
        if hasattr(layer, 'o_proj') and layer.o_proj is not None:
            # Prune output dimension
            layer.o_proj = self._prune_linear(layer.o_proj, keep_idxs, is_output=True)
            # Prune input dimension
            layer.o_proj = self._prune_linear(layer.o_proj, keep_idxs, is_output=False)
        
        # Update layer attributes
        layer.num_heads = new_num_heads
        layer.embed_dim = new_embed_dim if hasattr(layer, 'embed_dim') else new_embed_dim
        
        return layer
    
    def prune_in_channels(self, layer: nn.Module, idxs: Sequence[int]) -> nn.Module:
        """Prune input channels (same as output for attention due to residual)."""
        return self.prune_out_channels(layer, idxs)
    
    def get_out_channels(self, layer: nn.Module) -> int:
        """Get output dimension."""
        if hasattr(layer, 'embed_dim'):
            return layer.embed_dim
        elif hasattr(layer, 'num_heads') and hasattr(layer, 'head_dim'):
            return layer.num_heads * layer.head_dim
        return 0
    
    def get_in_channels(self, layer: nn.Module) -> int:
        """Get input dimension."""
        return self.get_out_channels(layer)
    
    def _prune_linear(self, linear: nn.Linear, keep_idxs: List[int], is_output: bool = True) -> nn.Linear:
        """Prune linear layer along specified dimension."""
        if linear is None:
            return None
        
        keep_idxs_tensor = torch.tensor(keep_idxs, device=linear.weight.device)
        
        if is_output:
            # Prune output dimension (dim 0)
            linear.out_features = len(keep_idxs)
            linear.weight = nn.Parameter(torch.index_select(linear.weight.data, 0, keep_idxs_tensor))
            if linear.bias is not None:
                linear.bias = nn.Parameter(torch.index_select(linear.bias.data, 0, keep_idxs_tensor))
        else:
            # Prune input dimension (dim 1)
            linear.in_features = len(keep_idxs)
            linear.weight = nn.Parameter(torch.index_select(linear.weight.data, 1, keep_idxs_tensor))
            # Bias doesn't change when pruning input
        
        return linear

In [10]:
# ============================================================
# 2. REGISTER THE CUSTOM PRUNER
# ============================================================

# Register SDPA pruner for Qwen2Attention
from transformers.models.qwen2.modeling_qwen2 import Qwen2Attention

# Create a mapping from module type to pruner
CUSTOM_PRUNER_MAP = {
    Qwen2Attention: SDPAPruner(),
}

# Monkey patch the PrunerBox to use our custom pruner for Qwen2Attention
original_pruner_box = tp.pruner.function.PrunerBox.copy()
tp.pruner.function.PrunerBox[Qwen2Attention] = SDPAPruner()

print("✓ Custom SDPA pruner registered")

✓ Custom SDPA pruner registered


In [11]:
class LayerWrapper(nn.Module):
    """Wrapper for single transformer layer to work with DependencyGraph."""
    
    def __init__(self, layer):
        super().__init__()
        self.layer = layer
    
    def forward(self, hidden_states, attention_mask=None, position_ids=None, **kwargs):
        # Handle tuple input
        if isinstance(hidden_states, tuple):
            hidden_states = hidden_states[0]
        
        # Call the layer with appropriate arguments
        if position_ids is not None:
            out = self.layer(
                hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                use_cache=False,
                output_attentions=False
            )
        else:
            out = self.layer(
                hidden_states,
                attention_mask=attention_mask,
                use_cache=False,
                output_attentions=False
            )
        
        # Extract hidden_states from output tuple
        if isinstance(out, tuple):
            out = out[0]
        
        return out

In [12]:

# Get first layer
layer0 = model.model.layers[0]
print(f"\nLayer 0 type: {type(layer0)}")
print(f"Attention module: {type(layer0.self_attn)}")

# Create wrapper
wrapped_layer = LayerWrapper(layer0).to(device)
wrapped_layer.eval()

# Prepare forward inputs for DependencyGraph
with torch.no_grad():
    # Get embeddings
    embeddings = model.get_input_embeddings()(example_inputs["input_ids"])
    
    # Create position_ids (required for Qwen2)
    position_ids = torch.arange(SEQ_LENGTH, device=device).unsqueeze(0).expand(BATCH_SIZE, -1)
    
    # Attention mask
    attention_mask = example_inputs["attention_mask"]

print("\n✓ Example inputs prepared")
print(f"  embeddings shape: {embeddings.shape}")
print(f"  attention_mask shape: {attention_mask.shape}")
print(f"  position_ids shape: {position_ids.shape}")

# Build DependencyGraph
print("\n🔨 Building DependencyGraph...")
DG = tp.DependencyGraph()



Layer 0 type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2DecoderLayer'>
Attention module: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2SdpaAttention'>

✓ Example inputs prepared
  embeddings shape: torch.Size([2, 64, 896])
  attention_mask shape: torch.Size([2, 64])
  position_ids shape: torch.Size([2, 64])

🔨 Building DependencyGraph...


In [14]:
try:
    # Try with all three arguments
    DG.build_dependency(
        wrapped_layer,
        example_inputs=(embeddings, attention_mask, position_ids)
    )
    print("✓ DependencyGraph built successfully with 3 arguments")
except Exception as e:
    print(f"Warning: Failed with 3 arguments: {e}")
    try:
        # Try with 2 arguments
        DG.build_dependency(
            wrapped_layer,
            example_inputs=(embeddings, attention_mask)
        )
        print("✓ DependencyGraph built successfully with 2 arguments")
    except Exception as e2:
        print(f"Warning: Failed with 2 arguments: {e2}")
        # Try with just embeddings
        DG.build_dependency(
            wrapped_layer,
            example_inputs=embeddings
        )
        print("✓ DependencyGraph built successfully with 1 argument")

✓ DependencyGraph built successfully with 3 arguments


/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['layer.input_layernorm.weight', 'layer.post_attention_layernorm.weight'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


In [15]:
# ============================================================
# 5. EXTRACT AND ANALYZE PRUNING GROUPS
# ============================================================

print("\n" + "="*70)
print("EXTRACTING PRUNING GROUPS")
print("="*70)

groups = []
group_info = {}

# Try to get all pruning groups
try:
    # Method 1: get_all_groups
    all_groups = list(DG.get_all_groups())
    print(f"Found {len(all_groups)} pruning groups via get_all_groups()")
    
    for group_idx, group in enumerate(all_groups[:20]):  # Limit to first 20
        group_size = len(group) if hasattr(group, '__len__') else 1
        groups.append(group)
        group_info[group_idx] = {
            'size': group_size,
            'type': 'dependency_group'
        }
        
except Exception as e:
    print(f"get_all_groups() failed: {e}")
    
    # Method 2: Create groups manually from attention module
    print("\nCreating groups manually from attention module...")
    
    attn_module = layer0.self_attn
    projection_names = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
    
    for proj_name in projection_names:
        if hasattr(attn_module, proj_name):
            proj = getattr(attn_module, proj_name)
            try:
                # Create a pruning group for this projection
                group = DG.get_pruning_group(
                    proj,
                    tp.prune_linear_out_channels,
                    idxs=[0]
                )
                groups.append(group)
                group_info[len(groups)-1] = {
                    'size': len(group),
                    'type': proj_name
                }
                print(f"  ✓ Created group for {proj_name}")
            except Exception as e2:
                print(f"  ✗ Failed to create group for {proj_name}: {e2}")

print(f"\nTotal groups created: {len(groups)}")

# ============================================================
# 6. DISPLAY GROUP INFORMATION
# ============================================================

print("\n" + "="*70)
print("GROUP DETAILS")
print("="*70)

for group_idx, group in enumerate(groups):
    print(f"\n--- Group {group_idx} ---")
    if group_idx in group_info:
        print(f"Type: {group_info[group_idx]['type']}")
        print(f"Size: {group_info[group_idx]['size']}")
    
    # Display group members
    if hasattr(group, '__iter__') and not isinstance(group, str):
        for i, item in enumerate(group[:5]):  # Show first 5
            if hasattr(item, '__getitem__') and len(item) >= 2:
                dep, idxs = item[0], item[1]
                if hasattr(dep, 'source') and hasattr(dep.source, 'module'):
                    src_module = type(dep.source.module).__name__
                    tgt_module = type(dep.target.module).__name__ if hasattr(dep, 'target') else 'unknown'
                    print(f"  [{i}] {src_module} -> {tgt_module}, idxs={idxs[:3]}...")
        if len(group) > 5:
            print(f"  ... and {len(group)-5} more items")


EXTRACTING PRUNING GROUPS
Found 5 pruning groups via get_all_groups()

Total groups created: 5

GROUP DETAILS

--- Group 0 ---
Type: dependency_group
Size: 19

--- Group 1 ---
Type: dependency_group
Size: 8

--- Group 2 ---
Type: dependency_group
Size: 4

--- Group 3 ---
Type: dependency_group
Size: 11

--- Group 4 ---
Type: dependency_group
Size: 38


# 1 layer vis

In [5]:
import torch
import plotly.graph_objects as go
import networkx as nx

In [6]:
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)

In [22]:
print("\n" + "="*60)
print("GROUPS (Pruning Groups)")
print("="*60)

for i, group in enumerate(groups):
    # 🔥 название группы
    root_module = group[0][0].source.module
    group_name = module_names.get(root_module, "unknown")

    print(f"\n--- Group {i} ---")
    print(f"Name: {group_name}")
    print(f"Root: {root_module.__class__.__name__}")
    print(f"Size: {len(group)}")

    for j, (dep, idxs) in enumerate(group):
        src = dep.source.module
        tgt = dep.target.module

        src_name = module_names.get(src, "unknown")
        tgt_name = module_names.get(tgt, "unknown")

        print(f"  [{j}]")
        print(f"    Source : {src_name} ({src.__class__.__name__})")
        print(f"    Target : {tgt_name} ({tgt.__class__.__name__})")
        print(f"    Handler: {dep.handler.__name__}")
        print(f"    Idxs   : {idxs}")


GROUPS (Pruning Groups)

--- Group 0 ---
Name: self_attn.q_proj
Root: Linear
Size: 38
  [0]
    Source : self_attn.q_proj (Linear)
    Target : self_attn.q_proj (Linear)
    Handler: prune_out_channels
    Idxs   : [0]
  [1]
    Source : self_attn.q_proj (Linear)
    Target : unknown (_ReshapeOp)
    Handler: prune_out_channels
    Idxs   : [0]
  [2]
    Source : unknown (_ReshapeOp)
    Target : unknown (_ElementWiseOp)
    Handler: prune_out_channels
    Idxs   : [0]
  [3]
    Source : unknown (_ElementWiseOp)
    Target : unknown (_SliceOp)
    Handler: prune_out_channels
    Idxs   : [0]
  [4]
    Source : unknown (_ElementWiseOp)
    Target : unknown (_SliceOp)
    Handler: prune_out_channels
    Idxs   : [0]
  [5]
    Source : unknown (_ElementWiseOp)
    Target : unknown (_ElementWiseOp)
    Handler: prune_out_channels
    Idxs   : [0]
  [6]
    Source : unknown (_ElementWiseOp)
    Target : unknown (_ElementWiseOp)
    Handler: prune_out_channels
    Idxs   : [0]
  [7]
    Sou

In [19]:
pos = nx.spring_layout(G, k=0.7, iterations=50)

# edges
edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=1),
    hoverinfo='none'
)

# nodes
node_x, node_y, node_text = [], [], []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=node_text,
    textposition="top center",
    hoverinfo='text',
    marker=dict(size=14)
)

fig = go.Figure(data=[edge_trace, node_trace])

fig.update_layout(
    title="Dependency Graph (Qwen2 + SDPA Attention)",
    showlegend=False
)

fig.show()

fig.write_html("sdpa_dependency_graph.html")

In [16]:
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title="Qwen2 Layer Dependency Graph (Parameter Groups)",
    showlegend=False
)

fig.show()

In [42]:
m = model.model.layers[0]
??m

Signature:       m(*args, **kwargs)
Type:            Qwen2DecoderLayer
String form:    
Qwen2DecoderLayer(
  (self_attn): Qwen2SdpaAttention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
    (rotary_emb): Qwen2RotaryEmbedding()
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLU()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)
File:            ~/projects/FedCore/.venv/lib/python3.10/site-packages/transformers/models/qwen2/modeling_qwen2.py
Source:         
class Qwen2DecoderLayer(nn.Module):
 

In [ ]:
torch.nn.functional.scaled_dot_product_attention

In [44]:
at = m.self_attn
??at

Signature:      at(*args, **kwargs)
Type:           Qwen2SdpaAttention
String form:   
Qwen2SdpaAttention(
  (q_proj): Linear(in_features=896, out_features=896, bias=True)
  (k_proj): Linear(in_features=896, out_features=128, bias=True)
  (v_proj): Linear(in_features=896, out_features=128, bias=True)
  (o_proj): Linear(in_features=896, out_features=896, bias=False)
  (rotary_emb): Qwen2RotaryEmbedding()
)
File:           ~/projects/FedCore/.venv/lib/python3.10/site-packages/transformers/models/qwen2/modeling_qwen2.py
Source:        
class Qwen2SdpaAttention(Qwen2Attention):
    """
    Qwen2 attention module using torch.nn.functional.scaled_dot_product_attention. This module inherits from
    `Qwen2Attention` as the weights of the module stays untouched. The only changes are on the forward pass to adapt to
    SDPA API.
    """

    # Adapted from Qwen2Attention.forward
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = 

# DEEPSEEK GRAPH VIS

In [56]:
BATCH_SIZE = 2
SEQ_LENGTH = 32
MODEL_NAME = "google/gemma-2-2b"
# MODEL_NAME = "Qwen/Qwen2.5-0.5B"
# MODEL_NAME = "arnir0/Tiny-LLM"

In [57]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype = torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded")

Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.01s/it]


Model loaded


In [30]:
example_inputs = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device),
    "attention_mask": torch.ones(BATCH_SIZE, SEQ_LENGTH).to(device),
    "labels": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device)
}

In [59]:
import torch
import torch.nn as nn
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
from typing import Dict, List, Optional, Any, Union
import torch_pruning as tp
import re


def extract_all_groups_with_modules(model: nn.Module, 
                                    DG: tp.DependencyGraph,
                                    root_module_types: List = None) -> Dict[int, List[nn.Module]]:
    """
    Extract all dependency groups and map them to actual modules.
    """
    if root_module_types is None:
        root_module_types = [nn.Linear, nn.Embedding, nn.Conv2d]
    
    groups = {}
    
    try:
        if hasattr(DG, 'get_all_groups'):
            all_groups = list(DG.get_all_groups(ignored_layers=[], root_module_types=root_module_types))
        else:
            all_groups = list(DG.get_all_groups())
    except Exception as e:
        print(f"Warning: Could not get groups: {e}")
        return groups
    
    for group_idx, group in enumerate(all_groups):
        modules_in_group = []
        
        for item in group:
            module = None
            
            if hasattr(item, 'module') and item.module is not None:
                module = item.module
            elif hasattr(item, 'target') and item.target is not None:
                if isinstance(item.target, str):
                    parts = item.target.split('.')
                    current = model
                    for part in parts:
                        if hasattr(current, part):
                            current = getattr(current, part)
                        else:
                            break
                    else:
                        module = current
            
            if module is not None and isinstance(module, nn.Module):
                modules_in_group.append(module)
        
        if modules_in_group:
            groups[group_idx] = modules_in_group
    
    return groups


def get_layer_order_key(name: str) -> tuple:
    """
    Extract ordering information from layer name for sequential layout.
    """
    if 'embed_tokens' in name or 'wte' in name or 'embedding' in name:
        return (0, 0, name)
    
    layer_match = re.search(r'layers?\.(\d+)', name)
    if layer_match:
        layer_num = int(layer_match.group(1))
        
        if 'self_attn' in name or 'attention' in name:
            return (1, layer_num, 0, name)
        elif 'input_layernorm' in name or 'norm1' in name:
            return (1, layer_num, 1, name)
        elif 'mlp' in name or 'ffn' in name:
            return (1, layer_num, 2, name)
        elif 'post_attention_layernorm' in name or 'norm2' in name:
            return (1, layer_num, 3, name)
        else:
            return (1, layer_num, 4, name)
    
    if 'lm_head' in name or 'output' in name or 'classifier' in name:
        return (2, 999, name)
    
    return (3, 0, name)


def get_module_name(model: nn.Module, target_module: nn.Module) -> str:
    """Get the name of a module within the model."""
    for name, module in model.named_modules():
        if module is target_module:
            return name
    return f"{type(target_module).__name__}_{id(target_module)}"


def get_group_info(model: nn.Module, example_inputs: Union[torch.Tensor, Dict]) -> Dict:
    """
    Get group information without creating visualization.
    """
    DG = tp.DependencyGraph()
    
    if isinstance(example_inputs, dict):
        DG.build_dependency(model, example_inputs=example_inputs)
    else:
        DG.build_dependency(model, example_inputs=example_inputs)
    
    groups = extract_all_groups_with_modules(model, DG)
    return groups


def visualize_model_structure(model: nn.Module, 
                              example_inputs: Union[torch.Tensor, Dict],
                              width: int = 1400,
                              height: int = 900) -> go.Figure:
    """
    Create an interactive Plotly diagram with working group selection dropdown.
    """
    
    # Build dependency graph
    DG = tp.DependencyGraph()
    
    if isinstance(example_inputs, dict):
        DG.build_dependency(model, example_inputs=example_inputs)
    else:
        DG.build_dependency(model, example_inputs=example_inputs)
    
    # Extract dependency groups
    groups = extract_all_groups_with_modules(model, DG)
    
    # Build graph for visualization
    G = nx.DiGraph()
    
    # Collect all prunable layers
    layers_info = []
    
    if len(groups) > 0:
        for group_id, modules in groups.items():
            for module in modules:
                module_name = get_module_name(model, module)
                layers_info.append((module_name, module, group_id))
    else:
        print("Collecting prunable layers from model...")
        for name, module in model.named_modules():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                group_id = -1
                layer_match = re.search(r'layers?\.(\d+)', name)
                if layer_match:
                    group_id = int(layer_match.group(1))
                layers_info.append((name, module, group_id))
    
    # Sort layers by order (top to bottom)
    layers_with_order = []
    for name, module, group_id in layers_info:
        order_key = get_layer_order_key(name)
        layers_with_order.append((order_key, name, module, group_id))
    
    layers_with_order.sort(key=lambda x: x[0])
    
    # Add nodes to graph
    num_layers = len(layers_with_order)
    pos = {}
    node_groups = {}  # Store group_id for each node
    
    for idx, (_, name, module, group_id) in enumerate(layers_with_order):
        # Calculate position
        x_pos = 0
        if 'embed' in name.lower():
            x_pos = 0
        elif 'q_proj' in name or 'k_proj' in name or 'v_proj' in name:
            x_pos = -1.5
        elif 'o_proj' in name:
            x_pos = 1.5
        elif 'gate_proj' in name or 'up_proj' in name:
            x_pos = -1
        elif 'down_proj' in name:
            x_pos = 1
        elif 'layernorm' in name.lower() or 'norm' in name.lower():
            x_pos = 0
        elif 'lm_head' in name or 'output' in name:
            x_pos = 0
        else:
            x_pos = 0
        
        y_pos = 1 - (2 * idx / max(1, num_layers - 1))
        pos[name] = (x_pos, y_pos)
        
        try:
            param_count = sum(p.numel() for p in module.parameters())
        except:
            param_count = 0
        
        module_type = type(module).__name__
        
        G.add_node(name,
                  group=group_id,
                  module_type=module_type,
                  param_count=param_count,
                  order_idx=idx,
                  short_name=name.split('.')[-1])
        
        node_groups[name] = group_id
    
    # Add edges
    for i in range(len(layers_with_order) - 1):
        current_name = layers_with_order[i][1]
        next_name = layers_with_order[i+1][1]
        G.add_edge(current_name, next_name, weight=1.0)
    
    for name in G.nodes():
        if 'attn' in name and 'v_proj' in name:
            base_name = name.replace('v_proj', 'o_proj')
            if base_name in G:
                G.add_edge(name, base_name, weight=0.5, style='dash')
    
    # Prepare common data
    node_names_list = list(G.nodes())
    node_x = [pos[node][0] for node in node_names_list]
    node_y = [pos[node][1] for node in node_names_list]
    node_short_names = [G.nodes[node].get('short_name', node) for node in node_names_list]
    
    # Prepare hover texts
    node_texts = []
    for node in node_names_list:
        group_id = G.nodes[node].get('group', -1)
        module_type = G.nodes[node].get('module_type', 'Unknown')
        param_count = G.nodes[node].get('param_count', 0)
        node_texts.append(f"<b>{module_type}</b><br>"
                         f"<b>Layer: {node}</b><br>"
                         f"Group: {group_id}<br>"
                         f"Parameters: {param_count:,}<br>"
                         f"Order: {G.nodes[node].get('order_idx', -1)}")
    
    # Color palette
    colorscale = px.colors.qualitative.Set3 + px.colors.qualitative.Plotly
    
    # Get unique groups
    unique_groups = sorted(set([g for g in node_groups.values() if g >= 0]))
    
    # Create edge traces (same for all views)
    edge_x_solid = []
    edge_y_solid = []
    edge_x_dashed = []
    edge_y_dashed = []
    
    for edge in G.edges():
        if edge[0] in pos and edge[1] in pos:
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            
            edge_data = G.get_edge_data(edge[0], edge[1])
            style = edge_data.get('style', 'solid')
            
            if style == 'dashed':
                edge_x_dashed.extend([x0, x1, None])
                edge_y_dashed.extend([y0, y1, None])
            else:
                edge_x_solid.extend([x0, x1, None])
                edge_y_solid.extend([y0, y1, None])
    
    edge_trace_solid = go.Scatter(
        x=edge_x_solid,
        y=edge_y_solid,
        mode='lines',
        line=dict(width=2, color='rgba(80,80,80,0.6)'),
        hoverinfo='none',
        name='Sequential Flow'
    )
    
    edge_trace_dashed = go.Scatter(
        x=edge_x_dashed,
        y=edge_y_dashed,
        mode='lines',
        line=dict(width=2, color='rgba(255,100,100,0.4)', dash='dash'),
        hoverinfo='none',
        name='Skip/Attention Connections'
    )
    
    # Create different node traces for each highlighting mode
    traces = [edge_trace_solid, edge_trace_dashed]
    buttons = []
    
    # 1. Default trace (no highlighting)
    default_colors = []
    default_sizes = []
    
    for node in node_names_list:
        group_id = node_groups[node]
        
        if 'embed' in node.lower():
            default_colors.append('#4CAF50')
            default_sizes.append(30)
        elif 'lm_head' in node.lower():
            default_colors.append('#FF9800')
            default_sizes.append(30)
        elif 'q_proj' in node or 'k_proj' in node or 'v_proj' in node:
            default_colors.append('#2196F3')
            default_sizes.append(30)
        elif 'o_proj' in node:
            default_colors.append('#3F51B5')
            default_sizes.append(30)
        elif 'gate_proj' in node or 'up_proj' in node:
            default_colors.append('#9C27B0')
            default_sizes.append(30)
        elif 'down_proj' in node:
            default_colors.append('#E91E63')
            default_sizes.append(30)
        elif 'norm' in node.lower():
            default_colors.append('#FFC107')
            default_sizes.append(30)
        else:
            if group_id >= 0:
                default_colors.append(colorscale[group_id % len(colorscale)])
            else:
                default_colors.append('#9E9E9E')
            default_sizes.append(25)
    
    default_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=node_short_names,
        textposition="top center",
        textfont=dict(size=10, color='black'),
        hovertext=node_texts,
        hoverinfo='text',
        marker=dict(
            size=default_sizes,
            color=default_colors,
            line=dict(width=2, color='#333333'),
            opacity=0.85,
            symbol='circle'
        ),
        name='All Layers',
        visible=True
    )
    traces.append(default_trace)
    
    # Add button for default view
    buttons.append(dict(
        args=[{"visible": [True, True] + [False] * (len(unique_groups) + 1)}],
        label="Show All (No Highlight)",
        method="restyle"
    ))
    
    # 2. Create a trace for each group highlight
    for target_group in unique_groups:
        highlight_colors = []
        highlight_sizes = []
        
        for node in node_names_list:
            group_id = node_groups[node]
            
            if group_id == target_group:
                # Highlight this group
                highlight_colors.append('#FF0000')  # Bright red
                highlight_sizes.append(45)  # Larger size
            else:
                # Dim other groups
                if group_id >= 0:
                    # Use muted version of the color
                    base_color = colorscale[group_id % len(colorscale)]
                    highlight_colors.append(base_color)
                else:
                    highlight_colors.append('#D3D3D3')  # Light gray
                highlight_sizes.append(20)  # Smaller size
        
        highlight_trace = go.Scatter(
            x=node_x,
            y=node_y,
            mode='markers+text',
            text=node_short_names,
            textposition="top center",
            textfont=dict(size=10, color='black'),
            hovertext=node_texts,
            hoverinfo='text',
            marker=dict(
                size=highlight_sizes,
                color=highlight_colors,
                line=dict(width=2, color='#333333'),
                opacity=0.85,
                symbol='circle'
            ),
            name=f'Group {target_group} Highlighted',
            visible=False
        )
        traces.append(highlight_trace)
        
        # Add button for this group
        # Create visibility list: edges always visible, only this trace visible
        visibility = [True, True]  # edges always visible
        for i in range(len(unique_groups) + 1):
            if i == 0:  # default trace
                visibility.append(False)
            elif i == target_group + 1:  # this group's highlight trace
                visibility.append(True)
            else:
                visibility.append(False)
        
        buttons.append(dict(
            args=[{"visible": visibility}],
            label=f"Highlight Group {target_group}",
            method="restyle"
        ))
    
    # Count total parameters
    total_params = sum(G.nodes[node].get('param_count', 0) for node in G.nodes())
    total_params_mb = total_params * 4 / (1024**2)
    
    # Create figure
    fig = go.Figure(data=traces,
                   layout=go.Layout(
                       title=dict(
                           text=f"<b>Model Structure Visualization with Group Selection</b><br>"
                                f"<sub>{type(model).__name__} - {G.number_of_nodes()} prunable layers, "
                                f"{total_params:,} parameters (~{total_params_mb:.1f} MB)</sub>",
                           font=dict(size=18),
                           x=0.5
                       ),
                       showlegend=True,
                       legend=dict(
                           x=0.02,
                           y=0.98,
                           bgcolor='rgba(255,255,255,0.8)',
                           bordercolor='black',
                           borderwidth=1
                       ),
                       hovermode='closest',
                       margin=dict(b=100, l=50, r=50, t=120),
                       xaxis=dict(
                           showgrid=True,
                           gridwidth=1,
                           gridcolor='rgba(200,200,200,0.3)',
                           zeroline=False,
                           showticklabels=False,
                           title="Layer Type Distribution",
                           range=[-2.5, 2.5]
                       ),
                       yaxis=dict(
                           showgrid=True,
                           gridwidth=1,
                           gridcolor='rgba(200,200,200,0.3)',
                           zeroline=False,
                           showticklabels=False,
                           title="Layer Order (Top → Bottom)",
                           autorange='reversed'
                       ),
                       plot_bgcolor='rgba(250,250,250,0.95)',
                       paper_bgcolor='rgba(250,250,250,0.95)',
                       height=height,
                       width=width,
                       updatemenus=[
                           dict(
                               buttons=buttons,
                               direction="down",
                               showactive=True,
                               x=0.02,
                               y=1.12,
                               xanchor="left",
                               yanchor="top",
                               pad={"r": 10, "t": 10},
                               bgcolor="white",
                               bordercolor="black",
                               borderwidth=2,
                               font=dict(size=12)
                           )
                       ],
                       annotations=[
                           dict(
                               text="🔽 SELECT GROUP TO HIGHLIGHT FROM DROPDOWN 🔽",
                               showarrow=False,
                               xref="paper",
                               yref="paper",
                               x=0.02,
                               y=1.06,
                               font=dict(size=12, color="red", weight="bold"),
                               bgcolor='rgba(255,255,255,0.9)',
                               bordercolor='red',
                               borderwidth=2
                           ),
                           dict(
                               text=f"📊 Available Groups: {unique_groups} | "
                                    f"🎨 Highlighted = Red & Larger | Others = Dimmed & Smaller",
                               showarrow=False,
                               xref="paper",
                               yref="paper",
                               x=0,
                               y=-0.12,
                               font=dict(size=11),
                               bgcolor='white',
                               bordercolor='gray',
                               borderwidth=1,
                               align='left'
                           ),
                           dict(
                               text=f"💡 Tip: Each group represents layers that are pruned together",
                               showarrow=False,
                               xref="paper",
                               yref="paper",
                               x=0,
                               y=-0.08,
                               font=dict(size=10, color="blue"),
                               bgcolor='rgba(255,255,255,0.8)',
                               align='left'
                           )
                       ]
                   ))
    
    return fig, unique_groups


def print_group_info(model: nn.Module, example_inputs: Union[torch.Tensor, Dict]):
    """
    Print detailed information about dependency groups.
    """
    groups = get_group_info(model, example_inputs)
    
    print("\n" + "="*70)
    print("DEPENDENCY GROUP INFORMATION")
    print("="*70)
    
    if len(groups) == 0:
        print("\n⚠️  No dependency groups found. Groups will be based on layer indices.")
        print("   This is normal for transformer models without explicit pruning groups.")
        
        # Collect layer indices
        layer_indices = set()
        for name, module in model.named_modules():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                layer_match = re.search(r'layers?\.(\d+)', name)
                if layer_match:
                    layer_indices.add(int(layer_match.group(1)))
        
        if layer_indices:
            print(f"\n📊 Layer indices found: {sorted(layer_indices)}")
            print(f"   Total distinct layer groups: {len(layer_indices)}")
    else:
        for group_idx, modules in groups.items():
            print(f"\n📦 Group {group_idx}:")
            print(f"   Contains {len(modules)} coupled layers:")
            for module in modules[:3]:  # Show first 3 modules
                module_name = get_module_name(model, module)
                module_type = type(module).__name__
                try:
                    param_count = sum(p.numel() for p in module.parameters())
                    print(f"      • {module_name} ({module_type}) - {param_count:,} params")
                except:
                    print(f"      • {module_name} ({module_type})")
            if len(modules) > 3:
                print(f"      ... and {len(modules) - 3} more modules")
        
        print(f"\n📊 Total groups: {len(groups)}")
    
    print("="*70)
    return groups


In [60]:
# Print group information first
groups = print_group_info(model, example_inputs)

# Create visualization with dropdown group selection
print("\n🎨 Creating interactive visualization with group selection...")
fig, available_groups = visualize_model_structure(
    model, 
    example_inputs,
    width=1400,
    height=900
)

# Show the figure
print("📊 Displaying visualization...")
fig.show()

# Save to HTML files
fig.write_html("model_visualization_with_dropdown.html")
print("\n✅ Visualization saved to model_visualization_with_dropdown.html")

# Also save with CDN for better performance
fig.write_html("model_visualization_interactive.html", 
               include_plotlyjs='cdn',
               full_html=True,
               config={'displayModeBar': True, 'responsive': True, 'scrollZoom': True})
print("✅ Interactive version saved to model_visualization_interactive.html")

# Print available groups
print(f"\n📊 Available Groups in this Model: {available_groups}")
print("   Use the dropdown menu in the top-left corner to select and highlight different groups")
print("   - 'Show All' displays all layers with default colors")
print("   - 'Highlight Group X' makes group X bright red/large and dims/shrinks all other groups")
print("\n💡 The dropdown will appear in the top-left corner of the visualization")

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.1.post_attention_layernorm.weight', 'model.layers.8.pre_feedforward_layernorm.weight', 'model.layers.10.post_attention_layernorm.weight', 'model.layers.22.post_attention_layernorm.weight', 'model.layers.6.post_attention_layernorm.weight', 'model.layers.11.post_feedforward_layernorm.weight', 'model.layers.2.pre_feedforward_layernorm.weight', 'model.layers.7.post_attention_layernorm.weight', 'model.layers.5.input_layernorm.weight', 'model.layers.6.input_layernorm.weight', 'model.layers.11.pre_feedforward_layernorm.weight', 'model.layers.12.pre_feedforward_layernorm.weight', 'model.layers.24.post_feedforward_layernorm.weight', 'model.layers.3.pre_feedforward_layernorm.weight', 'model.layers.5.pre_feedforward_layernorm.weight', 'model.layers.16.post_feedforward_layernorm.weight', 'model.layers.1.pre_feedforward_layernorm.weight', '

KeyboardInterrupt: 

In [61]:
import torch
import torch.nn as nn
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
from typing import Dict, List, Optional, Union
import torch_pruning as tp
import re
import time
import pickle
from pathlib import Path


class FastModelVisualizer:
    """Optimized visualizer for large models like Gemma2"""
    
    def __init__(self, cache_dir: str = "./vis_cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
    
    def get_layer_order_key(self, name: str) -> tuple:
        """Extract ordering information from layer name."""
        if 'embed_tokens' in name or 'wte' in name:
            return (0, 0, name)
        
        layer_match = re.search(r'layers?\.(\d+)', name)
        if layer_match:
            layer_num = int(layer_match.group(1))
            
            # Priority within layer
            priorities = {
                'input_layernorm': 0,
                'q_proj': 1, 'k_proj': 2, 'v_proj': 3,
                'self_attn': 4,
                'post_attention_layernorm': 5,
                'pre_feedforward_layernorm': 6,
                'gate_proj': 7, 'up_proj': 8,
                'down_proj': 9,
                'post_feedforward_layernorm': 10
            }
            
            for key, priority in priorities.items():
                if key in name:
                    return (1, layer_num, priority, name)
            return (1, layer_num, 99, name)
        
        if 'lm_head' in name or 'output' in name:
            return (2, 999, name)
        
        return (3, 0, name)
    
    def collect_layers_fast(self, model: nn.Module) -> List[tuple]:
        """Fast collection of prunable layers without full DG build."""
        layers_info = []
        
        for name, module in model.named_modules():
            # Only collect Linear and Embedding layers
            if isinstance(module, (nn.Linear, nn.Embedding)):
                try:
                    param_count = sum(p.numel() for p in module.parameters())
                    group_id = -1
                    
                    # Extract layer index if present
                    layer_match = re.search(r'layers?\.(\d+)', name)
                    if layer_match:
                        group_id = int(layer_match.group(1))
                    
                    layers_info.append((name, module, group_id, param_count))
                except:
                    pass
        
        return layers_info
    
    def build_graph_fast(self, layers_info: List[tuple]) -> nx.DiGraph:
        """Build graph quickly without heavy computations."""
        G = nx.DiGraph()
        
        # Sort layers by order
        layers_with_order = []
        for name, module, group_id, param_count in layers_info:
            order_key = self.get_layer_order_key(name)
            layers_with_order.append((order_key, name, module, group_id, param_count))
        
        layers_with_order.sort(key=lambda x: x[0])
        
        # Add nodes with positions (simplified)
        num_layers = len(layers_with_order)
        
        for idx, (_, name, module, group_id, param_count) in enumerate(layers_with_order):
            # Simplified positioning - much faster
            x_pos = 0
            if 'q_proj' in name or 'k_proj' in name or 'v_proj' in name:
                x_pos = -1.5
            elif 'o_proj' in name:
                x_pos = 1.5
            elif 'gate_proj' in name or 'up_proj' in name:
                x_pos = -1
            elif 'down_proj' in name:
                x_pos = 1
            elif 'norm' in name.lower():
                x_pos = 0
            
            y_pos = 1 - (2 * idx / max(1, num_layers - 1))
            
            G.add_node(name,
                      group=group_id,
                      module_type=type(module).__name__,
                      param_count=param_count,
                      x=x_pos,
                      y=y_pos,
                      short_name=name.split('.')[-1])
        
        # Add edges between consecutive layers
        for i in range(len(layers_with_order) - 1):
            current_name = layers_with_order[i][1]
            next_name = layers_with_order[i+1][1]
            G.add_edge(current_name, next_name)
        
        return G
    
    def create_figure_fast(self, model: nn.Module, example_inputs: Union[torch.Tensor, Dict],
                          width: int = 1400, height: int = 900, 
                          use_cache: bool = True) -> go.Figure:
        """
        Create visualization optimized for large models.
        """
        # Try to load from cache
        cache_file = self.cache_dir / f"{type(model).__name__}_graph.pkl"
        
        if use_cache and cache_file.exists():
            print("📦 Loading cached graph...")
            with open(cache_file, 'rb') as f:
                G = pickle.load(f)
        else:
            print("🔨 Building graph from model (this may take 15-30 seconds)...")
            start_time = time.time()
            
            # Fast collection without full dependency graph
            layers_info = self.collect_layers_fast(model)
            print(f"   Collected {len(layers_info)} prunable layers in {time.time()-start_time:.1f}s")
            
            # Build graph
            G = self.build_graph_fast(layers_info)
            print(f"   Graph built with {G.number_of_nodes()} nodes in {time.time()-start_time:.1f}s")
            
            # Cache the graph
            if use_cache:
                with open(cache_file, 'wb') as f:
                    pickle.dump(G, f)
                print(f"   Cached to {cache_file}")
        
        # Extract node data
        node_names = list(G.nodes())
        node_x = [G.nodes[n]['x'] for n in node_names]
        node_y = [G.nodes[n]['y'] for n in node_names]
        node_short_names = [G.nodes[n]['short_name'] for n in node_names]
        
        # Prepare hover texts
        node_texts = []
        for node in node_names:
            node_texts.append(f"<b>{G.nodes[node]['module_type']}</b><br>"
                            f"<b>{node}</b><br>"
                            f"Group: {G.nodes[node]['group']}<br>"
                            f"Parameters: {G.nodes[node]['param_count']:,}")
        
        # Get unique groups
        unique_groups = sorted(set([G.nodes[n]['group'] for n in node_names if G.nodes[n]['group'] >= 0]))
        
        # Create traces (same as before but optimized)
        traces = []
        buttons = []
        
        # Edge trace
        edge_x, edge_y = [], []
        for edge in G.edges():
            if edge[0] in G.nodes and edge[1] in G.nodes:
                x0, y0 = G.nodes[edge[0]]['x'], G.nodes[edge[0]]['y']
                x1, y1 = G.nodes[edge[1]]['x'], G.nodes[edge[1]]['y']
                edge_x.extend([x0, x1, None])
                edge_y.extend([y0, y1, None])
        
        edge_trace = go.Scatter(
            x=edge_x, y=edge_y, mode='lines',
            line=dict(width=1.5, color='rgba(80,80,80,0.4)'),
            hoverinfo='none', name='Data Flow'
        )
        traces.append(edge_trace)
        
        # Default trace (all layers)
        colorscale = px.colors.qualitative.Set3
        default_colors = []
        for node in node_names:
            group_id = G.nodes[node]['group']
            if 'embed' in node.lower():
                default_colors.append('#4CAF50')
            elif 'lm_head' in node.lower():
                default_colors.append('#FF9800')
            elif 'norm' in node.lower():
                default_colors.append('#FFC107')
            elif group_id >= 0:
                default_colors.append(colorscale[group_id % len(colorscale)])
            else:
                default_colors.append('#9E9E9E')
        
        default_trace = go.Scatter(
            x=node_x, y=node_y, mode='markers+text',
            text=node_short_names, textposition="top center",
            textfont=dict(size=9),
            hovertext=node_texts, hoverinfo='text',
            marker=dict(size=25, color=default_colors,
                       line=dict(width=1.5, color='#333'), opacity=0.85),
            visible=True
        )
        traces.append(default_trace)
        
        # Add button for default view
        buttons.append(dict(
            args=[{"visible": [True] + [False] * (len(unique_groups) + 1)}],
            label="Show All", method="restyle"
        ))
        
        # Create highlight traces for each group
        for target_group in unique_groups[:20]:  # Limit to 20 groups for performance
            highlight_colors = []
            highlight_sizes = []
            
            for node in node_names:
                if G.nodes[node]['group'] == target_group:
                    highlight_colors.append('#FF0000')
                    highlight_sizes.append(40)
                else:
                    highlight_colors.append('#D3D3D3')
                    highlight_sizes.append(18)
            
            highlight_trace = go.Scatter(
                x=node_x, y=node_y, mode='markers+text',
                text=node_short_names, textposition="top center",
                textfont=dict(size=9),
                hovertext=node_texts, hoverinfo='text',
                marker=dict(size=highlight_sizes, color=highlight_colors,
                           line=dict(width=1.5, color='#333'), opacity=0.85),
                visible=False
            )
            traces.append(highlight_trace)
            
            visibility = [True] + [False] * (len(unique_groups) + 1)
            visibility[target_group + 2] = True
            buttons.append(dict(
                args=[{"visible": visibility}],
                label=f"Group {target_group}", method="restyle"
            ))
        
        # Create figure
        total_params = sum(G.nodes[n]['param_count'] for n in node_names)
        
        fig = go.Figure(data=traces, layout=go.Layout(
            title=dict(text=f"<b>Gemma2 Model Structure</b><br>"
                           f"<sub>{G.number_of_nodes()} layers, {total_params:,} params</sub>",
                      font=dict(size=16), x=0.5),
            showlegend=False,
            margin=dict(b=80, l=50, r=50, t=100),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                      range=[-2.5, 2.5]),
            yaxis=dict(showgrid=True, gridwidth=0.5, 
                      gridcolor='rgba(200,200,200,0.3)',
                      zeroline=False, showticklabels=False,
                      title="Layer Order (Top → Bottom)", autorange='reversed'),
            height=height, width=width,
            updatemenus=[dict(buttons=buttons, direction="down", 
                             x=0.02, y=1.08, bgcolor="white",
                             bordercolor="black", borderwidth=1)],
            annotations=[
                dict(text="🔽 SELECT GROUP", x=0.02, y=1.04,
                     font=dict(size=11, color="red"), showarrow=False),
                dict(text=f"Groups: {len(unique_groups)}", x=0, y=-0.08,
                     font=dict(size=10), showarrow=False)
            ]
        ))
        
        return fig




In [62]:


# Быстрая визуализация
print("\n🎨 Creating optimized visualization...")
visualizer = FastModelVisualizer()
fig = visualizer.create_figure_fast(model, example_inputs, width=1200, height=800)

print("📊 Displaying...")
fig.show()

# Сохранение
fig.write_html("gemma2_visualization.html", config={'responsive': True})
print("✅ Saved to gemma2_visualization.html")


🎨 Creating optimized visualization...
🔨 Building graph from model (this may take 15-30 seconds)...
   Collected 184 prunable layers in 0.0s
   Graph built with 184 nodes in 0.0s
   Cached to vis_cache/Gemma2ForCausalLM_graph.pkl
📊 Displaying...


✅ Saved to gemma2_visualization.html
